# 第11章 注目する仕組み：Self-Attention

どの文字に注目するかを、文字どうしが話し合って決める

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：GPTの心臓部 **Self-Attention** を一行ずつ自作し、それだけで動く小さな言語モデルを学習させる
- **身につく数学・理論**：全位置で同時に「次」を予測する学習 ／ 過去の平均を行列の掛け算で一気に計算 ／ マスク付きsoftmax（未来を見ない） ／ 内積＝似ている度 ／ Query・Key・Value ／ $\sqrt{d}$ で割る理由（分散の話） ／ 位置埋め込み（Attentionは順番を知らない）
- **新しく使うPython・PyTorch**：`torch.tril` と `masked_fill` ／ `transpose` で行列の向きを変える ／ `register_buffer` ／ ブロードキャストつきの `@`

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 2時間 |
| 難易度 | ★★★★☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> 第9章のMLPは「直前8文字を横一列に並べて」見ていました。でも、この方法では窓を広げるほどパラメータが増え、しかも「どの文字が大事か」を場面ごとに変えられません。この章では、**場面に応じて、どの過去の文字に注目するかを自分で決める**仕組み **Self-Attention** を、ゼロから組み立てます。2017年の論文「Attention Is All You Need」で提案され、GPTを含む現代のほぼすべてのLLMの心臓部になっている仕組みです。

> **🔗 これまでの復習とつながり：これまでの復習（第3・6・9・10章とのつながり）**
>
> Self-Attention は一見むずかしそうですが、実はここまで学んだ4つの部品をそのまま組み合わせたものです。
>
> - **第3章（`softmax` と温度）**：生の点数（スコア）を「合計が1の正の重み」に変える仕組み。点数の差が大きすぎると1位だけ確率1に尖りすぎることも学びました。
> - **第6章（行列の掛け算 `@` とテンソルの形）**：`[M, K] @ [K, N] → [M, N]` のルールと、「for文を行列積にまとめると一気に速くなる」発想をフル活用します。
> - **第9章（文字の埋め込み `nn.Embedding`）**：文字IDを長さ `C` のベクトルへ変換し、`nn.Module` として層を組み立てました。
> - **第10章（足し算と分散の膨らみ）**：たくさんの数を足すとばらつき（分散）が大きくなる性質。これが「なぜ $\sqrt{d}$ で割るのか」の決め手になります。

**ミナト**：「注目する」って、人間みたいなことをするんですね。

**ボンゴ先生**：「先生は奥さんの顔を見て、彼女に……」の「彼女」が誰か分かるのは、キミが「奥さん」に注目したからだ。直前の「て、」を見ても分からない。どこを見るべきかは、文の内容しだいで変わる。それを、文字たちが自分たちで**話し合って決める**。それがAttentionだ。話し合いの方法は、意外なほど単純だよ。

## 準備と、新しい問題の出し方

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import re, math, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/0984f7dc18a2270bdfd2ff99bf14fd1b3ebb4f65/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    try:
        with urllib.request.urlopen(url, timeout=30) as res:
            text = res.read().decode("cp932")
    except Exception as e:
        raise RuntimeError(f"青空文庫の取得に失敗しました（時間をおいて再実行するか付録Eを参照）: {e}") from e
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])
n = int(len(data) * 0.9)
train_data, val_data = data[:n], data[n:]

def get_batch(data, batch_size, T, g):
    ix = torch.randint(0, len(data) - T, (batch_size,), generator=g)
    x = torch.stack([data[i:i + T] for i in ix])
    y = torch.stack([data[i + 1:i + T + 1] for i in ix])   # 1文字ずらしたもの
    return x, y

g = torch.Generator().manual_seed(0)
x, y = get_batch(train_data, 1, 8, g)
print("入力：", repr("".join(itos[i] for i in x[0].tolist())))
print("正解：", repr("".join(itos[i] for i in y[0].tolist())))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
入力： '私の注意を母より'
正解： 'の注意を母よりは'
```

</details>

第9章までと違うところに気づきましたか？ 今回は **正解も8文字** です。入力の8文字それぞれについて、「その次の文字」を予測させます。

| 位置 | 見てよい文字 | 予測する文字 |
|---|---|---|
| 0 | 私 | の |
| 1 | 私の | 注 |
| 2 | 私の注 | 意 |
| … | … | … |
| 7 | 私の注意を母より | は |

1つの窓から8問作れるので、効率がいいのです。ただし大事なルールがあります。**位置 t の予測には、位置 t までの文字しか使ってはいけない**。位置2で「意」を予測するときに、位置3にある「意」そのものを見たら、それはカンニングです。

<details><summary><b>基礎の補足：なぜ「1文字ずらし」で全位置同時に学習できるのか？</b>（クリックで開く）</summary>

第9章のMLPでは「8文字 → 次の1文字」という1問だけを作っていましたが、GPT系の言語モデルでは `x` と `y` を両方とも長さ `T` にして、**1文字ぶん右にずらして**並べます。

```python no-run
# 例：長さ T = 4 の窓を切り出すイメージ
text_ids = [10, 20, 30, 40, 50]
x = text_ids[0:4]   # [10, 20, 30, 40]  入力（位置 0, 1, 2, 3）
y = text_ids[1:5]   # [20, 30, 40, 50]  正解（各位置の「次の文字」）
# 位置 0: [10]             → 20 を当てる
# 位置 1: [10, 20]         → 30 を当てる
# 位置 2: [10, 20, 30]     → 40 を当てる
# 位置 3: [10, 20, 30, 40] → 50 を当てる
```

位置 `t` の正解 `y[t]` は、入力 `x[t + 1]` とまったく同じ文字です。だからこそ「未来の位置 `t + 1` 以降を見ない」というルール（因果マスク）が絶対条件になります。

なお、学習時にモデル自身の予測結果ではなく**正解のトークン列 `x` を全位置へ一気に与えて次トークンを同時に予測させる**この学習法を **教師強制（teacher forcing）** と呼びます（第8章の1文字ずらしの多位置版）。また、長さの異なる文をバッチにまとめる実務やチャット学習では、因果マスクに加えて「余白（パディング）を無視する **`attention_mask`**」と「質問部分の損失を除外する **損失マスク（`ignore_index=-100`）**」も併用します（第18章で実装します）。

</details>

#### 📝 確認問題（`ch11-q01` · Lv.1 · 全位置同時予測（1文字ずらし）とカンニング禁止ルール）

長さ $T = 8$ の入力 `x` が `'私の注意を母より'` のとき、正解 `y` は1文字ずらした `'の注意を母よりは'` になります。このとき、1つの窓から何問の「次の文字予測」を同時に学習でき、位置 `t = 2`（`'注'` の位置）ではどの範囲まで見てよいですか？

- **a.** 1問だけ学習でき、位置 `2` では後ろの `'意'` も含めて全 8 文字すべてを見てよい
- **b.** 8問同時に学習でき、どの位置でも文脈を広げるため全 8 文字を自由に見られる
- **c.** 8問同時に学習でき、位置 `2` では位置 `0〜2`（`'私の注'`）までしか見てはいけない
- **d.** 7問しか学習できず、過去が存在しない位置 `0` は常に損失計算から除外される

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：入力 `x` と正解 `y` はどちらも長さ 8 あり、位置 `t` ごとに「`x[0..t]` から `y[t]` を当てる」問題になっています。
- **ヒント 2**：位置 `2` の正解 `y[2]` は `'意'`（これは `x[3]` と同じ文字）です。カンニングを防ぐには位置何番まで見てよいか考えましょう。
- **正解**：`c`

**考え方**：言語モデルの学習では、長さ $T$ の1つの窓から「1文字見て次を当てる」〜「$T$ 文字見て次を当てる」までの $T$ 問を並列に解かせます。ただし位置 `t` は位置 `0..t` だけを見て予測する制約が必要です。

1. 入力 `x` は長さ $T = 8$、正解 `y` は `x` を1文字右にずらした長さ $8$。
1. 各位置 `t = 0, 1, ..., 7` で1問ずつ、計8問を同時に採点する。
1. 位置 `2` の正解は `y[2] = '意'`（これは `x[3]` と同じ）。カンニングを防ぐため見てよいのは `x[0..2]`（`'私の注'`）まで。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html#ex-ch11-q01

</details>

## ステップ1：過去を「平均」してみる

各位置が過去の情報を集める、いちばん簡単な方法は「**自分と、自分より前の全員の平均をとる**」ことです。埋め込みの代わりに、でたらめなベクトルで試してみます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
torch.manual_seed(0)
B, T, C = 4, 8, 2            # 4個の文章、8文字、各文字は2個の数
x = torch.randn(B, T, C)

xbow = torch.zeros(B, T, C)  # bow：bag of words（言葉の袋）
for b in range(B):
    for t in range(T):
        xbow[b, t] = x[b, :t + 1].mean(dim=0)   # 位置0〜tの平均

print(x[0])
print(xbow[0])

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
tensor([[-1.1258, -1.1524],
        [-0.2506, -0.4339],
        [ 0.8487,  0.6920],
        [-0.3160, -2.1152],
        [ 0.3223, -1.2633],
        [ 0.3500,  0.3081],
        [ 0.1198,  1.2377],
        [ 1.1168, -0.2473]])
tensor([[-1.1258, -1.1524],
        [-0.6882, -0.7931],
        [-0.1759, -0.2981],
        [-0.2109, -0.7524],
        [-0.1043, -0.8546],
        [-0.0286, -0.6608],
        [-0.0074, -0.3896],
        [ 0.1331, -0.3718]])
```

</details>

`xbow` の位置1は、`x` の位置0と1の平均（$(-1.1258 + -0.2506)/2 = -0.6882$）になっています。未来の文字は使っていません。

#### 📝 確認問題（`ch11-q02` · Lv.2 · 自分と過去の平均（Bag of Words）の計算）

未来を見ずに「位置 `0` から位置 `t` までの平均」をとる処理を確認します。次のコードを実行すると何が表示されますか？

```python
vals = [2.0, 6.0, 4.0, 8.0]
xbow = [sum(vals[:t + 1]) / (t + 1) for t in range(len(vals))]
print(xbow)
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：各位置 `t` では `vals[:t + 1]`（先頭から位置 `t` までの `t + 1` 個）の平均を順に計算します。
- **ヒント 2**：`t = 0` は `[2.0]`、`t = 1` は `[2.0, 6.0]`、`t = 2` は `[2.0, 6.0, 4.0]`、`t = 3` は全 4 個の平均をそれぞれ求めましょう。
- **期待される出力**：`[2.0, 4.0, 4.0, 5.0]`

**考え方**：位置 `t` の出力はスライス `vals[:t + 1]`（先頭から `t` 個目までの `t + 1` 個）の平均になり、未来の要素 `vals[t+1:]` は一切混ざりません。

1. `t=0`: `2.0 / 1 = 2.0`。
1. `t=1`: `(2.0 + 6.0) / 2 = 4.0`。
1. `t=2`: `(2.0 + 6.0 + 4.0) / 3 = 4.0`。
1. `t=3`: `(2.0 + 6.0 + 4.0 + 8.0) / 4 = 5.0`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html#ex-ch11-q02

</details>

## ステップ2：行列の掛け算で一気に

for文が二重になっていて遅そうです。第6章の教えを思い出しましょう。「for文を書いたら負け」。実は、この平均は**行列の掛け算1回**で計算できます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
wei = torch.tril(torch.ones(T, T))        # 下三角行列（左下だけ1）
wei = wei / wei.sum(dim=1, keepdim=True)  # 各行の合計が1になるように
print(wei)
xbow2 = wei @ x                           # [T, T] @ [B, T, C] → [B, T, C]
print(torch.allclose(xbow, xbow2))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
True
```

</details>

> **💡 理論のポイント：重みつき平均を行列で**
>
> `wei` の各行は「その位置が、どの位置の情報をどれだけの割合で集めるか」を表す**重み**です。行1は「位置0と1を半分ずつ」、行7は「8文字を1/8ずつ」。右上が0なのは「未来は見ない」からです。
>
> この重みの表 `wei` と `x` を掛けると、各位置について「重みつき平均」が一度に計算できます。`torch.tril`（triangle lower：下三角）は、行列の左下半分だけを残す関数です。
>
> さて、ここが大事な発想です。今は重みが**均等**（全員を同じ割合で平均）でした。でも本当は、「彼女」の位置では「奥さん」に大きな重みを、句読点には小さな重みを置きたい。**重みを内容に応じて変えられれば**、賢く情報を集められるはずです。

#### 📝 確認問題（`ch11-q03` · Lv.2 · 下三角行列 torch.tril と行列積 wei @ x による重みつき平均）

`wei` を形 `[T, T]` の下三角行列（各行の合計が1になるよう正規化済み）、`x` を形 `[B, T, C]` の入力テンソルとします。`wei @ x` を計算すると何が起き、結果の形は何になりますか？

- **a.** 軸の数が一致しないため行列積で必ずエラーになり、`wei` を事前に `[B, C, T]` へ変形する必要がある
- **b.** 二重の `for` 文なしで全バッチ・全位置の「過去の重みつき平均」が一括計算され、形は `[B, T, C]` になる
- **c.** 未来の位置も含めた全文字の重みつき合計が計算され、出力テンソルの形は `[B, T, T]` に変換される
- **d.** 各位置の過去ベクトルがすべて横一列に連結され、出力テンソルの形は `[B, T * C]` へ平坦化される

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：3次元テンソルとの行列積 `[T, T] @ [B, T, C]` では、先頭のバッチ軸 `B` がブロードキャストされ、内側の `T` が消えます。
- **ヒント 2**：`wei` の第 `t` 行は右上（未来）が 0 なので、`wei @ x` は二重ループなしで全バッチ・全位置の「過去の重みつき平均」を `[B, T, C]` で一括出力します。
- **正解**：`b`

**考え方**：下三角の重み行列 `wei` を左から掛けるだけで、未来を遮断した重みつき平均が全バッチ・全位置同時に高速計算できます。GPUが得意な行列積に落とし込むのがTransformer高速化の要です。

1. `wei` の形 `[T, T]` はブロードキャストにより `[B, T, T]` として扱われる。
1. `[B, T, T] @ [B, T, C]` → 内側の `T` が消えて `[B, T, C]`。
1. `wei` の第 `t` 行は右上が `0`、左下が `1/(t+1)` なので、`x[b, 0..t]` の平均に一致する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html#ex-ch11-q03

</details>

## ステップ3：softmaxで重みを作る

重みを自由に決められる形に書き直します。第3章のsoftmaxを使います。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
tril = torch.tril(torch.ones(T, T))
scores = torch.zeros(T, T)                                # 今はまだ全部0点
scores = scores.masked_fill(tril == 0, float("-inf"))     # 未来の位置は -∞ 点
wei = F.softmax(scores, dim=-1)
print(wei[:3])
print(torch.allclose(wei @ x, xbow))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])
True
```

</details>

> **💡 理論のポイント：なぜ 0 ではなく −∞ でマスクするのか？（因果マスク）**
>
> `masked_fill(tril == 0, float("-inf"))` は、「下三角の外（未来の位置）の点数を $-\infty$ にする」操作です。これを **因果マスク**（causal mask）と言います。カンニング防止の目隠しです。
>
> ここでよくある疑問が **「未来を消したいなら、点数を `0` にすればいいのでは？」** です。2トークンの小さな例（位置0の点数が `[1.0, 2.0]`、右側の位置1が未来）で手計算して比べてみましょう。
>
> - **失敗案：未来を `0.0` で埋める** → 点数は `[1.0, 0.0]`。softmax は $e^z$ を計算するため、$e^0 = 1 \neq 0$ になってしまいます！ 重みは $\left[\frac{e^1}{e^1 + e^0},\; \frac{e^0}{e^1 + e^0}\right] \approx [0.73,\; 0.27]$ となり、**未来の文字を27%もカンニング**してしまいます。
> - **正解：未来を $-\infty$ で埋める** → 点数は `[1.0, -inf]`。$e^{-\infty} = 0$ なので、重みは $\left[\frac{e^1}{e^1 + 0},\; \frac{0}{e^1 + 0}\right] = [1.00,\; 0.00]$ となり、未来の重みが**ぴったり0**になります。
>
> 点数が全部0なら、softmaxの結果は均等な重み、つまり平均と同じになりました。ここで、**点数（scores）を内容から計算する**ようにすれば、内容に応じた重みが作れます。どうやって点数をつけるか？ ここで「内積」の出番です。

#### 📝 確認問題（`ch11-q04` · Lv.2 · 因果マスクで 0 ではなく -∞ を使う理由とマスク無し時の問題）

因果マスク（causal mask）を `scores` にかけるとき、未来の位置を `0` ではなく **`float("-inf")`（$-\infty$）** で埋めるのはなぜですか？

- **a.** 未来の位置に $-\infty$ を入れると指数変換で重みが `1.0` に正規化され、将来の文脈を最も強く取り込んで高精度に学習できるから
- **b.** GPU の計算メモリ使用量を半分に削減するための実装上の最適化であり、softmax 後の数学的な重み自体は `0` を入れた場合とまったく同じだから
- **c.** `0` だと $e^0 = 1 \neq 0$ で未来に正の重みが残ってしまうが、$-\infty$ なら $e^{-\infty} = 0$ で未来の重みがぴったり 0 になるから
- **d.** PyTorch の `F.softmax` は入力テンソルに `0` が含まれていると対数計算で例外を出す仕様のため、代わりに $-\infty$ を入れる必要があるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：第3章の softmax の式 $p_i = e^{z_i} / \sum_j e^{z_j}$ において、$z_i = 0$ のとき $e^0$ はいくつになるか思い出しましょう。
- **ヒント 2**：$e^0 = 1 \neq 0$ なので、未来の点数を `0` にしても正の重みが残ってカンニングが起きます。重みを完全に `0` にするには $e^{-\infty} = 0$ が必要です。
- **正解**：`c`

**考え方**：softmax は入力を指数関数 $e^z$ に通します。$e^0 = 1$ なので `0` は「無視」ではなく「ふつうの点数0点」として重みをもらいます。重みを完全に0にするには $e^{-\infty} = 0$ となる $-\infty$ が必須です。

1. たとえば位置0の点数が `[1.0, 2.0]` で、位置1が未来とする。
1. 未来を `0.0` で埋めると `[1.0, 0.0]` → $e^1 \approx 2.72, e^0 = 1$ → 未来の重みは約 `27%` も残る！
1. 未来を `-inf` で埋めると `[1.0, -inf]` → $e^1 \approx 2.72, e^{-\infty} = 0$ → 未来の重みはぴったり `0%`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html#ex-ch11-q04

</details>

#### 📝 確認問題（`ch11-q05` · Lv.2 · コード予測：0埋めマスクと -inf マスクの softmax 出力の違い）

未来の位置を `0.0` で埋めた場合と `float("-inf")` で埋めた場合の softmax 結果（未来の重み）を比べます。次のコードの出力を予測してください。

```python
import math

def softmax(xs):
    m = max(xs)
    ex = [math.exp(x - m) for x in xs]
    s = sum(ex)
    return [round(e / s, 2) for e in ex]

print(softmax([0.0, 0.0]))
print(softmax([0.0, float("-inf")]))
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`softmax` は各要素の指数 $e^{x - m}$ の比率をとります。`math.exp(0.0)` と `math.exp(float('-inf'))` の値の違いに注目しましょう。
- **ヒント 2**：1行目の `[0.0, 0.0]` は `ex = [1.0, 1.0]` で均等割り、2行目の `[0.0, -inf]` は `ex = [1.0, 0.0]` になります。それぞれ合計で割ったリストを書きます。
- **期待される出力**：`[0.5, 0.5]
[1.0, 0.0]`

**考え方**：点数が同じ `0.0` のとき、未来を `0.0` で埋めたままにすると50%も未来を見てしまいます。`float("-inf")` なら `math.exp(-inf) == 0.0` なのでぴったり `[1.0, 0.0]` になります。

1. `softmax([0.0, 0.0])`: `ex = [1.0, 1.0]` → `[0.5, 0.5]`。
1. `softmax([0.0, -inf])`: `m = 0.0`、`ex = [1.0, 0.0]` → `[1.0, 0.0]`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html#ex-ch11-q05

</details>

## 内積：似ている度

#### 📐 数学のポイント：内積

2つのベクトルの、同じ位置どうしを掛けて全部足したものを **内積**（dot product）と言います。

$$\boldsymbol{a} \cdot \boldsymbol{b} = a_1 b_1 + a_2 b_2 + \cdots + a_d b_d$$

第6章の行列の掛け算は、実は「行と列の内積」をたくさん計算していたのでした。内積は、2つのベクトルが **同じ方向を向いているほど大きく**なります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
a = torch.tensor([1.0, 0.0])
print("同じ向き：", a @ torch.tensor([2.0, 0.0]))
print("直角：    ", a @ torch.tensor([0.0, 2.0]))
print("逆向き：  ", a @ torch.tensor([-2.0, 0.0]))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
同じ向き： tensor(2.)
直角：     tensor(0.)
逆向き：   tensor(-2.)
```

</details>

つまり内積は「どれくらい似ているか（向きがそろっているか）」の点数として使えます。純Pythonの `for` 文（Σ記号）で書けばたった1行です。

```python
# 内積 a・b = Σ (a_i * b_i) を素の Python で書く
a = [2.0, -1.0, 3.0]
b = [4.0,  2.0, 1.0]
dot = sum(x * y for x, y in zip(a, b))   # 2*4 + (-1)*2 + 3*1 = 9.0
```

#### 📝 確認問題（`ch11-q06` · Lv.2 · 内積の手計算と「向きの一致度」としての解釈）

2つのベクトル $\boldsymbol{q} = [3, -2, 4]$ と $\boldsymbol{k} = [2, 1, 3]$ の内積 $\boldsymbol{q} \cdot \boldsymbol{k}$ を計算してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：内積は2つのベクトルの「同じ位置の成分どうしを掛けて、すべて足し合わせる」計算です。
- **ヒント 2**：$3 \times 2 + (-2) \times 1 + 4 \times 3$ を計算し、正負の符号に気をつけて足し合わせましょう。
- **正解**：`16`

**考え方**：内積は対応する成分どうしの積の和 $\sum_i q_i k_i$ です。値が大きいほど2つのベクトルが同じ方向を向いており、Attentionでは高い注目スコアになります。

1. 1番目：$3 \times 2 = 6$。
1. 2番目：$(-2) \times 1 = -2$。
1. 3番目：$4 \times 3 = 12$。
1. 合計：$6 + (-2) + 12 = 16$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html#ex-ch11-q06

</details>

## ステップ4：Query・Key・Value

いよいよSelf-Attentionの本体です。各位置（各文字）は、自分のベクトルから3種類のベクトルを作ります。

> **💡 理論のポイント：図書館で調べものをする**
>
> Attentionは、図書館での調べものに似ています。
>
> - **Query（クエリ、質問）**：「私は何を探している？」。各文字が出す検索ワード
> - **Key（キー、ラベル）**：「私はこういう情報を持っています」。各文字が背表紙に貼るラベル
> - **Value（バリュー、中身）**：「私を選んでくれたら、これを渡します」。本の中身
>
> 位置 t の文字は、自分のQueryと、過去の各文字のKeyの **内積** を計算します。これが「この本は私の探しものにどれくらい合っているか」の点数です。点数をsoftmaxで重みにして、各文字の **Value** を重みつきで平均したものを受け取ります。
>
> 「彼女」がQueryで「女性の人物を探しています」と問い合わせ、「奥」がKeyで「女性の人物です」と答えれば、内積が大きくなり、「彼女」は「奥」のValueを多く受け取る。そんなイメージです。QueryとKeyとValueを作る変換（行列）は、もちろん **学習で決まります**。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
torch.manual_seed(0)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)      # [B, T, 16]
q = query(x)    # [B, T, 16]
scores = q @ k.transpose(-2, -1)    # [B, T, 16] @ [B, 16, T] → [B, T, T]
scores = scores.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(scores, dim=-1)
out = wei @ value(x)                # [B, T, T] @ [B, T, 16] → [B, T, 16]

print(scores.shape, wei.shape, out.shape)
print(wei[0].detach().round(decimals=2))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
torch.Size([4, 8, 8]) torch.Size([4, 8, 8]) torch.Size([4, 8, 16])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8800, 0.1200, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0600, 0.3700, 0.5800, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1700, 0.4500, 0.0400, 0.3400, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2800, 0.1800, 0.0500, 0.3100, 0.1800, 0.0000, 0.0000, 0.0000],
        [0.4700, 0.1400, 0.2700, 0.0300, 0.0500, 0.0500, 0.0000, 0.0000],
        [0.0500, 0.1800, 0.2500, 0.0100, 0.3800, 0.0800, 0.0500, 0.0000],
        [0.0900, 0.2700, 0.0100, 0.2100, 0.0200, 0.0900, 0.2400, 0.0700]])
```

</details>

重みが**均等ではなくなりました**。位置1は位置0に（0.88）、位置2は自分自身に（0.58）、位置5は位置0に（0.47）、と行ごとに注目する相手が違います。まだ学習していないので注目の仕方はでたらめですが、「内容に応じて重みが変わる」仕組みはこれで完成です。

#### ✨ コツ：形の早見表（迷ったらここに戻る）

Attentionで一番迷子になりやすいのは「形」です。B＝文章の数、T＝文字数、C＝埋め込みの大きさ、hs＝ヘッドの大きさ、として、

| もの | 形 | ひとこと |
|---|---|---|
| 入力 `x` | `[B, T, C]` | 文章ごと・文字ごとのベクトル |
| `q`, `k`, `v` | `[B, T, hs]` | 全結合層で小さくした「質問・ラベル・中身」 |
| `k.transpose(-2, -1)` | `[B, hs, T]` | 掛け算のために向きを変えたラベル |
| `scores`, `wei` | `[B, T, T]` | 行＝注目する文字、列＝注目される文字 |
| 出力 `wei @ v` | `[B, T, hs]` | 各文字が集めてきた情報 |

コードの途中で `print(〇〇.shape)` を入れて、この表と見比べると、どこで何が起きているかが分かります。

> **🐍 Python・PyTorchのポイント：transpose・reshape/view・detach**
>
> `k.transpose(-2, -1)` は、最後の2つの軸を入れ替えます。`[B, T, 16]` が `[B, 16, T]` になり、`q @ k.transpose(…)` で「全部のQueryと全部のKeyの内積」が `[B, T, T]` の表として一度に計算できます。行がQuery（注目する側）、列がKey（注目される側）です。<br>
> 3次元のテンソルどうしの `@` は、先頭の `B` の軸ごとに（4つの文章それぞれで）行列の掛け算をしてくれます。<br>
> **`.reshape` と `.view` の注意点**：`transpose` 直後のテンソルはメモリ上で不連続（non-contiguous）になるため、そのまま `.view(...)` を呼ぶと `RuntimeError` になります。形を変えたいときは `.reshape(...)`（または `.contiguous().view(...)`）を使うのが安全です（第6章・[付録B](https://sakunokyumei.github.io/llm-research-textbook/b-python.html)）。<br>
> **`.detach()` と `no_grad()` の違い**：`wei[0].detach()` は「値・形はそのままに、このテンソル1個だけを計算グラフ（勾配の記録）から切り離す」操作です。ブロック全体の記録を止める `with torch.no_grad():`、要素1個をPython数値にする `.item()` と目的に応じて使い分けます。

*(対話デモはWeb教科書ページで確認できます：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html)*

**ミナト**：デモで重みが変わるのは分かりました！ でも $QK^\top / \sqrt{d}$ の数式を、小さな数字で1回だけ手計算してみたいです。

**ボンゴ先生**：いい心がけだ！ たった2文字・2次元の小さな数字で最後まで手計算してみよう。一度手で追えば、Self-Attentionの式はもう怖くないよ。

#### 📐 数学のポイント：Scaled Dot-Product Attention：記号表と2トークン×2次元の完全手計算

Self-Attention の中心式を、記号の意味から2トークン×2次元の小さな数字による手計算まで一気に通しで確認しましょう。

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right) V$$

**① 記号の対応表（1文字ずつ確認）**

| 記号 | 形（1文章あたり） | 意味とコードとの対応 |
|---|---|---|
| $Q$ | `[T, d]` | **Query**（各位置が「何を探しているか」）。`q = query(x)` |
| $K$ | `[T, d]` | **Key**（各位置が「どんな情報を持っているか」）。`k = key(x)` |
| $K^\top$ | `[d, T]` | **$K$ の転置**（行と列を入れ替えたもの）。`k.transpose(-2, -1)` |
| $QK^\top$ | `[T, T]` | 全ペアの「Query と Key の内積」スコア表。`q @ k.transpose(-2, -1)` |
| $\sqrt{d}$ | スカラー | ヘッド次元 $d$（`head_size`）の平方根。内積の分散を約1に戻す係数（次節で詳しく） |
| $V$ | `[T, d]` | **Value**（選ばれたときに渡す情報の中身）。`v = value(x)` |

**② 2トークン・2次元（$T = 2,\; d = 2,\; \sqrt{2} \approx 1.414$）の完全手計算**

次の極小データを使って、最初から最後まで手で追ってみましょう。

$$Q = \begin{pmatrix} 1 & 0 \\ 0 & 2 \end{pmatrix}, \quad K = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}, \quad V = \begin{pmatrix} 10 & 0 \\ 0 & 20 \end{pmatrix}$$

1. **内積表 $QK^\top$ を計算する**（行が Query、列が Key）：
   - 位置0 × 位置0：$1\cdot 1 + 0\cdot 0 = 1$、位置0 × 位置1：$1\cdot 0 + 0\cdot 1 = 0$
   - 位置1 × 位置0：$0\cdot 1 + 2\cdot 0 = 0$、位置1 × 位置1：$0\cdot 0 + 2\cdot 1 = 2$
   $$QK^\top = \begin{pmatrix} 1 & 0 \\ 0 & 2 \end{pmatrix}$$
2. **$\sqrt{d} = \sqrt{2} \approx 1.414$ で割る**：
   $$\frac{QK^\top}{\sqrt{2}} \approx \begin{pmatrix} 0.707 & 0.000 \\ 0.000 & 1.414 \end{pmatrix}$$
3. **因果マスク（右上＝未来を $-\infty$ にする）**：
   $$\text{masked} \approx \begin{pmatrix} 0.707 & -\infty \\ 0.000 & 1.414 \end{pmatrix}$$
4. **各行ごとに `softmax` をかけて重み `wei` を作る**：
   - 行0（位置0）：$e^{-\infty} = 0$ なので $[1.00,\; 0.00]$（過去がないので100%自分自身）
   - 行1（位置1）：$e^{0} = 1.00,\; e^{1.414} \approx 4.11$（合計 $5.11$）→ $[1.00/5.11,\; 4.11/5.11] \approx [0.20,\; 0.80]$
5. **重み `wei` で $V$ を重みつき平均する（`wei @ V`）**：
   - 位置0の出力：$1.00 \cdot [10, 0] + 0.00 \cdot [0, 20] = [10.0,\; 0.0]$
   - 位置1の出力：$0.20 \cdot [10, 0] + 0.80 \cdot [0, 20] \approx [1.96,\; 16.09]$

位置1は「位置0から20%、位置1（自分）から80%」の割合で Value をブレンドした新しいベクトルを受け取りました。これが、バッチ $B$・長さ $T$ になってもまったく同じ計算で行われます。

#### 📝 確認問題（`ch11-q07` · Lv.2 · Query・Key・Value のそれぞれの役割）

Self-Attention における **Query・Key・Value** の役割と計算の組み合わせとして、正しいものはどれですか？

- **a.** **Query**・**Key**・**Value** は人間が辞書や文法規則をもとに手作業で作る固定テーブルであり、誤差逆伝播による学習では一切更新されない
- **b.** 注目される側の **Key** と中身を表す **Value** の内積から注目重みを計算し、その重みを使って質問側の **Query** を重みつき平均して出力する
- **c.** **Query** は未来の文字のベクトル、**Key** は過去の文字のベクトル、**Value** は現在位置の文字のベクトルだけを表すように役割が固定されている
- **d.** 注目する側の **Query** と注目される側の **Key** の内積から重み（Attention weights）を計算し、その重みで **Value** を重みつき平均して出力する

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：図書館のたとえ（検索ワード・本の背表紙ラベル・本の中身）と、Q・K・V の3つの対応を思い出しましょう。
- **ヒント 2**：「誰に注目するか」は Query と Key の内積＋softmax で決め、最後に集めて混ぜ合わせる情報の中身が Value です。
- **正解**：`d`

**考え方**：「どの文字に注目するか」を決める照合ステップ（$QK^\top$）と、「注目した先からどんな情報を引き出すか」（$V$）を別々の変換行列に分けることで、柔軟な情報収集が可能になります。

1. 各位置の `x` から `q = query(x)`、`k = key(x)`、`v = value(x)` を作る。
1. 照合：`scores = q @ k.transpose(-2, -1) / sqrt(d)` → マスク → `wei = softmax(scores)`。
1. 情報取得：`out = wei @ v`（Value の重みつき平均）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html#ex-ch11-q07

</details>

#### 📝 確認問題（`ch11-q08` · Lv.3 · Self-Attention 内部のテンソル shape トレース）

バッチサイズ $B = 4$、文字数 $T = 8$、埋め込み次元 $C = 32$、ヘッド次元 $hs = 16$ のとき、Self-Attention 内の `k.transpose(-2, -1)`、`scores`（`q @ k.transpose(-2, -1)`）、および出力 `out`（`wei @ v`）の形の組み合わせとして正しいものはどれですか？

- **a.** `k.transpose(-2, -1)`: `[16, 8, 4]`、`scores`: `[4, 16, 16]`、`out`: `[4, 8, 32]`
- **b.** `k.transpose(-2, -1)`: `[4, 8, 16]`、`scores`: `[4, 8, 8]`、`out`: `[4, 8, 32]`
- **c.** `k.transpose(-2, -1)`: `[4, 16, 8]`、`scores`: `[4, 8, 8]`、`out`: `[4, 8, 16]`
- **d.** `k.transpose(-2, -1)`: `[4, 16, 8]`、`scores`: `[4, 8, 16]`、`out`: `[4, 8, 8]`

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`q, k, v` はいずれも `[B, T, hs] = [4, 8, 16]` です。`transpose(-2, -1)` は最後の 2 軸だけを入れ替えます。
- **ヒント 2**：`k.transpose(-2, -1)` は `[4, 16, 8]`。行列積 `[4, 8, 16] @ [4, 16, 8]` で `scores`、さらに `@ [4, 8, 16]` で `out` の形を追いましょう。
- **正解**：`c`

**考え方**：Attention で迷子にならない最大のコツは、`scores` と `wei` が「文章ごとの全トークン対全トークン表 `[B, T, T]`」になり、最後に `v: [B, T, hs]` を掛けて `[B, T, hs]` に戻る流れを押さえることです。

1. `q`, `k`, `v`：`[4, 8, 16]`。
1. `k.transpose(-2, -1)`：最後の2軸を入れ替えて `[4, 16, 8]`。
1. `scores = q @ k^T`：`[4, 8, 16] @ [4, 16, 8]` → `[4, 8, 8]`（`wei` も同じ `[4, 8, 8]`）。
1. `out = wei @ v`：`[4, 8, 8] @ [4, 8, 16]` → `[4, 8, 16]`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html#ex-ch11-q08

</details>

*(この節の追加問題 `ch11-q09` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/11-attention.html で自動採点・復習できます)*

## なぜ √d で割るのか

実際のAttentionでは、点数を $\sqrt{d}$（$d$ はhead_size）で割ります。これを忘れるとどうなるか、実験してみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
torch.manual_seed(0)
for d in [16, 64, 256]:
    q = torch.randn(1000, d)
    k = torch.randn(1000, d)
    dots = (q * k).sum(dim=1)        # 1000組の内積
    print(f"d={d:3d}：内積の分散 {dots.var():7.2f}   √dで割ると {(dots / math.sqrt(d)).var():.2f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
d= 16：内積の分散   16.00   √dで割ると 1.00
d= 64：内積の分散   63.10   √dで割ると 0.99
d=256：内積の分散  259.85   √dで割ると 1.02
```

</details>

#### 💡 理論のポイント：点数が大きすぎると、softmaxが尖りすぎる（分散のスケーリング仮定）

**理想的な初期化時の仮定**として、$q_i, k_i$ が互いに独立で平均0・分散1だとすると、積 $q_i k_i$ の分散は 1 になり、それを $d$ 個足した内積の分散はちょうど $d$ 倍に膨らみます（実際の学習済み重みでは厳密に独立・分散1ではないため、これは設計の根拠を与える**仮定つきのスケーリング則**です）。

$$\mathrm{Var}\!\left(\sum_{i=1}^{d} q_i k_i\right) = d \quad \Longrightarrow \quad \mathrm{Var}\!\left(\frac{1}{\sqrt{d}}\sum_{i=1}^{d} q_i k_i\right) = \frac{d}{(\sqrt{d})^2} = 1$$

割らないままだと、たとえば $d = 256$ では標準偏差が $\sqrt{256} = 16$ にもなります。点数の差が大きすぎると、softmax の結果がほぼ「1位だけ `1.0`、残りは `0.0`」（one-hot）に尖ってしまいます（第3章の温度が低すぎる状態）。

そうなると、ほかの位置に注目する可能性が最初から閉ざされてしまいます。さらに困ったことに、softmax は確率が 1 や 0 に近づくと**傾き（微分）がほぼ 0 になる**（飽和する）性質があるため、誤差逆伝播で勾配が伝わらなくなり、学習が完全に止まってしまいます。$\sqrt{d}$ で割れば分散が約 1 に戻り、ほどよい温度になって勾配もしっかり流れます。これを **Scaled Dot-Product Attention**（スケールつき内積アテンション）と言い、式で書くとこうなります。

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right) V$$

この式が、論文「Attention Is All You Need」の中心にある式です。あなたはもう、この式のすべての記号を説明できます。

#### 📝 確認問題（`ch11-q10` · Lv.3 · QK^T を √d で割る理由（分散増大・softmax飽和・勾配消失））

Scaled Dot-Product Attention の式 $\mathrm{softmax}(QK^\top / \sqrt{d})V$ で、なぜ $QK^\top$ を $\sqrt{d}$ で割るのですか？ 割らないと何が起きますか？

- **a.** 行列 `q` と `k.transpose(-2, -1)` を掛け合わせたときに内側の次元が一致せずエラーになるのを防ぎ、テンソル形状を `[B, T, T]` に揃えるから
- **b.** 次元 $d$ が大きくなると内積の計算結果が必ず大きな負の数に偏ってしまうため、$\sqrt{d}$ で割って正のスコア範囲へ戻す必要があるから
- **c.** 次元 $d$ が大きいほど内積の分散が約 $d$ 倍に膨らみ、割らないと softmax がほぼ one-hot に飽和して勾配がほぼ 0 になり学習が止まるから
- **d.** 未来のトークン位置に付いた不要なスコアを 0 に打ち消すためのスケーリングであり、$-\infty$ による因果マスクの代わりを果たしているから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：平均 0・分散 1 の独立な数同士の積を $d$ 個足すと、内積のばらつき（分散）は $d$ 倍に膨らみます。点数の差が極端に開くと softmax はどうなるでしょう？
- **ヒント 2**：点数の差が大きすぎると softmax が 1 点集中（ほぼ one-hot）に尖り、傾き（勾配）がほぼ 0 に飽和して学習が止まります。$\sqrt{d}$ で割ると分散が 1 に戻ります。
- **正解**：`c`

**考え方**：独立な標準正規分布の積の和は分散が $d$（標準偏差が $\sqrt{d}$）になります。$\sqrt{d}$ で割ることで分散を $d / (\sqrt{d})^2 = 1$ に保ち、softmax の温度が極端に低くなる（飽和して勾配が消える）のを防ぎます。

1. $q_i, k_i$ が平均0・分散1なら、積 $q_i k_i$ の分散も1。
1. $d$ 個足すと分散は $d$、標準偏差は $\sqrt{d}$ に膨らむ。
1. $\sqrt{d}$ で割ると分散は $d / d = 1$ に戻る。
1. 点数のばらつきが適度に保たれ、softmax の勾配が消えずに学習が進む。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html#ex-ch11-q10

</details>

#### 📝 確認問題（`ch11-q11` · Lv.3 · head_size に応じたスケーリング係数 √d の計算）

`head_size`（$d$）が `64` のとき、$QK^\top$ の点数をいくつ（$\sqrt{d}$）で割りますか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：Scaled Dot-Product Attention では、内積の分散 $d$ を 1 に戻すため $d$ の平方根（$\sqrt{d}$）で割ります。
- **ヒント 2**：2乗して `64` になる正の整数（$x^2 = 64$ を満たす $x$）を求めましょう。
- **正解**：`8`

**考え方**：`head_size = 64` のとき $\sqrt{64} = 8$ で割ることで、内積の分散（約64）を $64 / 8^2 = 1$ に戻します。

1. $d = 64$。
1. $\sqrt{d} = \sqrt{64} = 8$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html#ex-ch11-q11

</details>

## 部品にまとめる：Head クラス

ここまでを1つのクラスにまとめます。これが **Attentionヘッド** です。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        self.wei = F.softmax(scores, dim=-1)       # 後で中身を見られるように取っておく
        return self.wei @ v

> **🐍 Python・PyTorchのポイント：register_buffer**
>
> `register_buffer` は「学習しないけれど、モデルの一部として持っておきたいテンソル」を登録する命令です。マスク用の下三角行列 `tril` は学習で変わる数ではありませんが、モデルと一緒にGPUへ移動してほしいので、こうして登録します。

本当に未来を見ていないか、**カンニング検査**をしてみましょう。入力の最後の文字だけを書き換えて、それより前の位置の出力が変わらないことを確かめます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
torch.manual_seed(0)
head = Head(n_embd=32, head_size=16, block_size=8)
x1 = torch.randn(1, 8, 32)
x2 = x1.clone()                       # x1 は残したまま、別のコピーを作る（x2 = x1 だけだと同じものを指す）
x2[0, 7] = torch.randn(32)            # 位置7（最後）だけ書き換える

out1, out2 = head(x1), head(x2)
print("位置0〜6の出力は同じ？", torch.allclose(out1[0, :7], out2[0, :7]))
print("位置7の出力は同じ？  ", torch.allclose(out1[0, 7], out2[0, 7]))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
位置0〜6の出力は同じ？ True
位置7の出力は同じ？   False
```

</details>

未来（位置7）を変えても、過去（位置0〜6）の出力はまったく変わりません。カンニングしていない証拠です。

ここで、PyTorch なしの素の Python でも Scaled Dot-Product Attention を自力実装して、仕組みを完全に定着させましょう！

#### 📝 確認問題（`ch11-q12` · Lv.2 · register_buffer・tril[:T, :T] の切り出し・カンニング検査）

`Head` クラスで下三角マスク `tril` を `self.register_buffer("tril", ...)` で登録し、`forward` の中で `self.tril[:T, :T]` と切り出している理由の組み合わせとして正しいものはどれですか？

- **a.** `register_buffer` は `tril` を学習可能パラメータとして勾配降下で更新し未来の参照割合を最適化するため、`:T, :T` はバッチサイズ `B` の違いを吸収して揃えるため
- **b.** `register_buffer` は不要になった中間テンソルを即座に削除して GPU メモリを解放するため、`:T, :T` は入力文の最後の文字を切り捨てて 1 文字短い行列を作るため
- **c.** `register_buffer` は CPU 実行では必須ではない単なる飾りであり、`self.tril[:T, :T]` も入力は常に `T == block_size` で固定されるため実は省略してもエラーにならない
- **d.** `register_buffer` は「学習で更新しないがモデルと一緒に GPU 移動・保存したいテンソル」として登録するため、`:T, :T` は `block_size` より短い入力長 `T` にも対応するため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：マスク行列 `tril` は学習で変えたい値でしょうか、固定したい値でしょうか？ また文章生成の開始時（例：`'私は'` で `T = 2`）の入力長と `block_size` の関係を考えましょう。
- **ヒント 2**：`register_buffer` は学習対象外のまま `.to(device)` や `state_dict()` に追従させる仕組みで、`self.tril[:T, :T]` は `T < block_size` の短い入力にもサイズを合わせる切り出しです。
- **正解**：`d`

**考え方**：モデル内の固定テンソル（マスクなど）を `register_buffer` に登録しておくと、学習で壊されることなく `.to(device)` や `state_dict()` に自動で追従します。また `[:T, :T]` の切り出しにより任意の長さ `T <= block_size` で動きます。

1. `tril` は固定マスクなのでパラメータではなく buffer として登録する。
1. `model.to(device)` したとき、`register_buffer` したテンソルも一緒にGPUへ移る。
1. 入力の長さ `T` は毎回 `block_size` 未満になり得るため、`self.tril[:T, :T]` で形 `[T, T]` に合わせる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html#ex-ch11-q12

</details>

#### 📝 確認問題（`ch11-q13` · Lv.4 · 【自力実装】純Pythonで内積＋スケール＋因果マスク付きAttention重み）

PyTorch を使わず純Python（標準ライブラリ `math` のみ）で、2次元リスト `Q`（形 `[T, d]`）と `K`（形 `[T, d]`）から、スケール（$/\sqrt{d}$）と因果マスク（`j > i` は `-inf`）を適用した Attention 重み行列 `wei`（形 `[T, T]`）を返す関数 `causal_attention_weights(Q, K)` を実装してください。

**ひな形コード：**
```python
import math

def causal_attention_weights(Q, K):
    T = len(Q)
    d = len(Q[0])
    scale = math.sqrt(d)
    wei = []
    for i in range(T):
        # 1. 各 j (0..T-1) について:
        #    j > i なら float("-inf")、そうでなければ Q[i] と K[j] の内積を scale で割る
        # 2. その行 scores_row にオーバーフロー安全な softmax をかけて wei に追加
        pass
    return wei
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：各行 `i` について列 `j` を 0 から `T - 1` まで回し、未来（`j > i`）を `-inf`、過去・自分（`j <= i`）を内積 `/ sqrt(d)` にしてから行ごとに安全な softmax をかけます。
- **ヒント 2**：`j > i` なら `float('-inf')`、それ以外は `sum(a * b for a, b in zip(Q[i], K[j])) / scale` を集め、`m = max(scores_row)` を引いて指数化・正規化しましょう。

**解答例：**
```python
import math

def causal_attention_weights(Q, K):
    T = len(Q)
    d = len(Q[0])
    scale = math.sqrt(d)
    wei = []
    for i in range(T):
        scores_row = []
        for j in range(T):
            if j > i:
                scores_row.append(float("-inf"))
            else:
                dot = sum(a * b for a, b in zip(Q[i], K[j]))
                scores_row.append(dot / scale)
        m = max(scores_row)
        exps = [math.exp(s - m) for s in scores_row]
        total = sum(exps)
        wei.append([e / total for e in exps])
    return wei
```

**検算テスト（Colabで実行して確認）：**
```python
Q = [[1.0, 0.0], [0.0, 2.0]]
K = [[1.0, 0.0], [0.0, 1.0]]
W = causal_attention_weights(Q, K)
assert len(W) == 2 and len(W[0]) == 2
assert abs(W[0][0] - 1.0) < 1e-9 and W[0][1] == 0.0
assert abs(sum(W[1]) - 1.0) < 1e-9
expected_w11 = math.exp(math.sqrt(2)) / (1.0 + math.exp(math.sqrt(2)))
assert abs(W[1][1] - expected_w11) < 1e-6
# すべて0なら均等な下三角平均になることも確認
Z = [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]
W0 = causal_attention_weights(Z, Z)
assert abs(W0[2][0] - 1/3) < 1e-9 and W0[0][2] == 0.0
print("ALL PASSED")
```

**考え方**：行列演算ライブラリの中身を素の Python ループで書ければ、「内積 → $\sqrt{d}$ スケーリング → 上三角を $-\infty$ でマスク → 行方向 softmax」という Self-Attention の心臓部が完全に自分のものになります。

1. `scale = math.sqrt(len(Q[0]))` を求める。
1. 各行 `i` について、`j <= i` なら `dot(Q[i], K[j]) / scale`、`j > i` なら `float("-inf")`。
1. 行ごとに最大値 `m` を引いて `math.exp(s - m)` をとり、行和で割る。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html#ex-ch11-q13

</details>

*(この節の追加問題 `ch11-q14` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/11-attention.html で自動採点・復習できます)*

## Attentionだけで言語モデルを作る

Headを1つだけ使って、言語モデルを作ってみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
class OneHeadLM(nn.Module):
    def __init__(self, V, block_size, n_embd=64):
        super().__init__()
        self.tok = nn.Embedding(V, n_embd)           # 文字の埋め込み
        self.pos = nn.Embedding(block_size, n_embd)  # 位置の埋め込み（後で説明）
        self.head = Head(n_embd, n_embd, block_size)
        self.out = nn.Linear(n_embd, V)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T))
        x = x + self.head(x)              # 過去の情報を「足し込む」
        logits = self.out(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

@torch.no_grad()
def val_loss(model, T):
    model.eval()
    xs = torch.stack([val_data[i:i + T] for i in range(0, len(val_data) - T - 1, T)])
    ys = torch.stack([val_data[i + 1:i + T + 1] for i in range(0, len(val_data) - T - 1, T)])
    loss = model(xs, ys)[1].item()
    model.train()
    return loss

block_size = 64
torch.manual_seed(0)
g = torch.Generator().manual_seed(0)
model = OneHeadLM(V, block_size)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
print("パラメータの数：", sum(p.numel() for p in model.parameters()))

for step in range(2001):
    x, y = get_batch(train_data, 32, block_size, g)
    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        print(f"ステップ {step:4d}：訓練 {loss.item():.3f}  検証 {val_loss(model, block_size):.3f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
パラメータの数： 282253
ステップ    0：訓練 7.958  検証 7.851
ステップ  500：訓練 3.294  検証 3.542
ステップ 1000：訓練 3.170  検証 3.364
ステップ 1500：訓練 2.950  検証 3.316
ステップ 2000：訓練 2.774  検証 3.312
```

</details>

> **🐍 Python・PyTorchのポイント：logits.view(B * T, -1)**
>
> 今までの `F.cross_entropy` は、`[問題の数, V]` の点数と `[問題の数]` の正解を受け取りました。今回の `logits` は `[B, T, V]`（文章 × 位置 × 語彙）と、1次元多いですね。そこで `logits.view(B * T, -1)` で `[B×T, V]` に並べ直し、正解も `targets.view(B * T)` で `[B×T]` にします。「32個の文章 × 64か所」＝2048問を、1列に並べてまとめて採点しているのです。`-1` は「残りの数は自動で計算して」という意味です。

> **💡 理論のポイント：位置の埋め込み：Attentionは順番を知らない（順列等変性）**
>
> Attentionの計算（内積→softmax→重みつき平均）をよく見ると、**文字の順番や距離の情報がどこにもありません**。数学的には「入力の並び替えに対して区別がつかない（**順列等変** / permutation-equivariant）」性質を持っています。たとえば「犬が猫を噛んだ」と「猫が犬を噛んだ」でも、位置情報がなければ各文字の Query・Key・Value はまったく同じになり（因果マスクで見える範囲が変わるだけ）、語順が区別できません。
>
> そこで、文字の埋め込みに **位置の埋め込み**（positional embedding）を足します。「0番目」「1番目」…という位置ごとにも学習可能なベクトルを用意して、文字のベクトルに足し合わせるのです。`self.pos(torch.arange(T))` がそれです。こうすると、同じ「私」でも「0番目の私」と「5番目の私」で少しずつ違うベクトルになり、語順や距離を学べるようになります。
>
> 最新のLLMでは、もっと賢い **RoPE**（回転位置埋め込み）という方法が使われています。第25章で、自分のGPTに移植します。

> **💡 理論のポイント：足し込む（残差接続）**
>
> `x = x + self.head(x)` で、Attentionの結果を元の `x` に**足して**います。元の情報（自分自身が何の文字か）を残したまま、過去から集めた情報を上乗せするためです。この「足し込む」構造を **残差接続**（residual connection）と言い、第12章で大活躍します。

パラメータ28万個、Headたった1つで、検証損失 **3.31前後**。第9章のMLP（66万個、3.395）より少ないパラメータで、良い成績です。しかもこのモデルは、**64文字前まで**見ることができます。窓を64に広げても、Attentionの部分のパラメータは1つも増えていません（位置の埋め込みが少し増えるだけ）。これがAttentionの強みです。

| モデル | 章 | パラメータ | 検証データの損失 |
|---|---|---|---|
| bigram（数える） | 5 | — | 3.581 |
| MLP（SGD） | 9 | 66万 | 3.395 |
| MLP（全部の治療） | 10 | 66万 | 3.177 |
| **Attention 1ヘッド（2000ステップ）** | **11** | **28万** | **3.31前後** |

1ヘッドでは、まだ第10章の治療済みMLPには届きません。でも、ここに部品（複数のヘッド、MLP、正規化、残差接続、層の積み重ね）を足していくと……それが次の章のTransformerです。

#### 📝 確認問題（`ch11-q15` · Lv.3 · 位置埋め込みが必要な理由と logits.view(B * T, -1)）

Self-Attention を使う言語モデルで、文字の埋め込み `self.tok(idx)` に加えて**位置の埋め込み** `self.pos(torch.arange(T))` を足すのはなぜですか？

- **a.** 各文字に絶対位置の番号を割り当てて重複をまとめることで、モデルが扱う語彙数 $V$ を大幅に削減して出力層の計算を軽くできるから
- **b.** 文字埋め込み単体では特徴ベクトルの次元数が半分しか足りないため、位置埋め込みを横に連結して次元数を 2 倍へ拡張する必要があるから
- **c.** 位置埋め込みを足しておけば未来の位置と過去の位置をモデルが自動判別できるため、$-\infty$ による因果マスクを省略できるようになるから
- **d.** Attention の計算（内積と重みつき平均）自体には文字の順番・距離の情報が含まれず、位置埋め込みがないと語順の違いを区別できなくなるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「犬が猫を噛んだ」と「猫が犬を噛んだ」では登場する文字の集合は同じですが順番が違います。内積と重みつき平均の式に位置インデックス $t$ は入っているでしょうか？
- **ヒント 2**：Attention 自体は集合（袋）に対する計算で順序の概念を持たないため、各位置のベクトル `self.pos(torch.arange(T))` を足して語順を区別できるようにします。
- **正解**：`d`

**考え方**：Self-Attention は入力トークン集合のペアごとの内積で重みを決めるため、本質的に「順列に対して等変（順序の概念がない）」です。位置埋め込みを足すことで初めて「何番目の文字か」「どれくらい離れているか」を区別できます。

1. Attention の式 $\mathrm{softmax}(QK^\top/\sqrt{d})V$ には位置インデックス $t$ が登場しない。
1. 位置ごとに学習可能なベクトル `self.pos(torch.arange(T))` を用意して `self.tok(idx)` に足す。
1. 同じ文字でも「0番目の私」と「5番目の私」が異なるベクトルになり、語順を学べる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html#ex-ch11-q15

</details>

## 注目の様子をのぞく

学習したHeadが、実際にどこに注目しているかを見てみましょう。

📦 **準備セル**：ライブラリや教材ファイルを取得します。エラーが出た場合は `-q` を外して詳細を確認してください。

In [ ]:
!pip install -q matplotlib-fontja

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

sample = "私はその人を常に先生と呼んでいた。"
idx = torch.tensor([[stoi[c] for c in sample]])
model.eval()
with torch.no_grad():
    model(idx)
wei = model.head.wei[0]

plt.figure(figsize=(6, 5))
plt.imshow(wei, cmap="Oranges")
plt.xticks(range(len(sample)), list(sample))
plt.yticks(range(len(sample)), list(sample))
plt.xlabel("注目される文字（Key）")
plt.ylabel("注目する文字（Query）")
plt.title("学習したAttentionの重み")
plt.show()

![学習したAttentionの重みのヒートマップ。各行で、自分自身や少し前の文字に重みが集中している](https://sakunokyumei.github.io/llm-research-textbook/img/figs/11-attention/cell11_0.png)

各行が「その文字が、どの文字にどれだけ注目したか」です。右上が真っ白なのは、マスクで未来を見ていないからです。このヘッドは、自分自身よりも**少し前の特定の文字**に強く注目しています。たとえば「常」「に」は「を」に、「で」は「呼」や「先」に注目しています。人間には意味の分かりにくい注目の仕方ですが、モデルは損失を下げるのに役立つ見方を、自分で見つけ出したのです。それでも、1つのヘッドにできることは限られています。次の章では、ヘッドをたくさん用意して、それぞれに違う役割（直前を見る係、文の主語を探す係……）を分担させます。

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **`RuntimeError: The size of tensor a (8) must match the size of tensor b (64)`**：マスク `tril` の大きさと、入力の長さ `T` が合っていません。`self.tril[:T, :T]` のように、入力の長さに合わせて切り出しましょう。
> - **損失が `nan`**：マスクで全部 $-\infty$ の行ができると、softmaxが計算できず `nan` になります。下三角（対角線を含む）を使えば、どの行にも少なくとも自分自身があるので大丈夫です。
> - **`IndexError: index out of range in self`**（`self.pos` の行）：`block_size` より長い入力を入れています。生成するときは、入力を最後の `block_size` 文字に切り詰めましょう（第12章）。

#### 📝 確認問題（`ch11-q16` · Lv.4 · Self-Attention が窓を広げても強い理由と1ヘッドの限界）

第9章のMLPと比べて、Self-Attention は「窓の長さ `block_size` を8文字から64文字へ広げたとき」にどんな強みがありますか？ また、それでもヘッドが1つ（`OneHeadLM`）だけでは不十分な理由もあわせて説明してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：MLP が窓を広げると入力層の重み数がどう変わるかと、Attention の Q/K/V 行列の共有・動的な注目先、さらに 1 枚の注目行列 `wei` で表現できる観点の数を整理しましょう。
- **ヒント 2**：①MLP は `T` に比例して入力重みが増え注目位置も固定だが、Attention は Q/K/V を全位置で共有するためパラメータを増やさず文脈に応じて柔軟に注目できる、②1 ヘッドでは 1 種類の注目パターンしか同時に表せないため複数観点の分業にマルチヘッドが必要、の 2 点をまとめます。
- **解答の目安**：第9章のMLPは過去 `T` 文字の埋め込みをすべて横に連結して全結合層に渡すため、窓を広げるほど入力層の重みパラメータ数が `T` に比例して急増し、位置ごとに固定の重みしか学べませんでした。一方 Self-Attention は、すべての位置で同じ Q・K・V 変換行列を共有して内容に応じた重みで情報を集めるため、窓を64文字に広げても Attention 本体のパラメータ数は増えず、必要な過去の文字に柔軟に注目できます。ただしヘッドが1つだけだと「1種類の注目パターン」しか同時に表せないため、直前の文字・主語・括弧の対応など複数の観点を並行して追うには複数のヘッド（第12章のマルチヘッドAttention）が必要です。

**考え方**：「同じ Q/K/V 行列で長い文脈を動的に参照できる」のが Self-Attention の最大のブレークスルーであり、「複数の観点で同時に参照したい」という動機が第12章のマルチヘッドAttentionへ直結します。

1. MLPとの比較：MLPは窓を広げると連結ベクトルが伸びて重みが激増し、注目位置も固定。
1. Attentionの強み：Q・K・V行列は位置によらず共通なので窓を広げてもAttention部分のパラメータは増えず、文脈に応じて注目先を変えられる。
1. 1ヘッドの限界：1枚の重み行列 `wei` では1種類の見方しか同時にできない → 次章で複数ヘッドに分業させる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html#ex-ch11-q16

</details>

**✅ できたらチェック**

- [ ] 「過去の平均」を、for文と行列の掛け算の両方で計算できた
- [ ] 因果マスク（$-\infty$ を入れてsoftmax）の意味を説明できる
- [ ] 内積が「似ている度」になることを確かめた
- [ ] Query・Key・Valueの役割を、自分の言葉で説明できる
- [ ] $\sqrt{d}$ で割る理由を実験で確かめた
- [ ] カンニング検査で、未来を見ていないことを確かめた
- [ ] 1ヘッドのAttention言語モデルを学習させた

> **📝 まとめ**
>
> - 言語モデルは窓の **全位置で同時に** 次の文字を予測する。ただし未来は見ない（**因果マスク**）
> - Self-Attention ＝ 内容に応じた **重みつき平均**。重みは **Query** と **Key** の内積、混ぜる中身は **Value**
> - 点数は $\sqrt{d}$ で割る：$\mathrm{softmax}(QK^\top/\sqrt{d})V$
> - Attentionは順番を知らないので、**位置の埋め込み** を足す
> - 窓を広げても、Attention部分のパラメータは増えない

**ボンゴ先生**：キミは今、2017年に世界を変えた式を、自分の手で書いて動かした。次の章では、このHeadを何個も並べ、MLPと正規化と残差接続でサンドイッチにして、何層も積み上げる。それが **Transformer**。GPTの「T」だ。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/11-attention.html
- **次の章**：[第12章 部品を組み立てる：Transformer](https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html)